# EDA & ML — DataLake Météo

Exploration des données Silver (échantillon d'une partition) et démonstration d'un premier modèle de prévision de température (XGBoost). Ce notebook lit un échantillon via l'API WebHDFS (``requests``), sans Spark, puis prépare un mini-modèle ML.

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

NAMENODE = "http://namenode:9870/webhdfs/v1"
USER = "root"


def webhdfs_list(path):
    resp = requests.get(NAMENODE + path, params={"op": "LISTSTATUS", "user.name": USER}, timeout=60)
    resp.raise_for_status()
    return resp.json()["FileStatuses"]["FileStatus"]


def webhdfs_download_file(remote_path, local_path):
    resp = requests.get(NAMENODE + remote_path, params={"op": "OPEN", "user.name": USER}, timeout=120)
    resp.raise_for_status()
    with open(local_path, "wb") as fh:
        fh.write(resp.content)


def webhdfs_download_dir(remote_dir, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    files = [e["pathSuffix"] for e in webhdfs_list(remote_dir) if e["pathSuffix"].endswith(".parquet")]
    frames = []
    for name in files:
        local = os.path.join(local_dir, name)
        webhdfs_download_file(remote_dir + "/" + name, local)
        frames.append(pd.read_parquet(local))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# Premier répertoire dt= listé sous /silver/meteo
partitions = sorted(e["pathSuffix"] for e in webhdfs_list("/silver/meteo") if e["pathSuffix"].startswith("dt="))
print("Partitions Silver disponibles (extrait) :", partitions[:5])
first_partition = partitions[0]
print("Partition choisie :", first_partition)

sample_dir = tempfile.mkdtemp(prefix="silver_sample_")
df = webhdfs_download_dir("/silver/meteo/" + first_partition, sample_dir)
print("Échantillon chargé :", df.shape)

In [ ]:
print("Aperçu du DataFrame :")
display(df.head())
print()
print("Shape :", df.shape)
print()
print("Types (dtypes) :")
display(df.dtypes)
print()
print("Valeurs manquantes :")
display(df.isna().sum())

In [ ]:
print("Statistiques descriptives de la température :")
display(df["temperature"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["temperature"].dropna(), kde=True, ax=axes[0])
axes[0].set_title("Distribution de la température")
group_col = "source" if df["source"].nunique() <= 10 else "city"
sns.boxplot(data=df, x=group_col, y="temperature", ax=axes[1])
axes[1].set_title("Température par " + group_col)
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [c for c in ["temperature", "precipitation", "wind_speed", "snow"] if c in df.columns]
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Corrélation température / précipitations / vent")
plt.show()

# Série temporelle moyenne par ville (top 8 villes de l'échantillon)
ts = df.copy()
ts["timestamp"] = pd.to_datetime(ts["timestamp"])
ts = ts.sort_values("timestamp")
top_cities = df["city"].value_counts().head(8).index
for city in top_cities:
    grp = ts[ts["city"] == city]
    grp.set_index("timestamp")["temperature"].plot(label=city)
plt.legend()
plt.title("Température par ville (échantillon)")
plt.show()

## Pourquoi XGBoost ?

Pour ce TP de prévision de température à J+1, **XGBoost** (gradient boosting sur arbres) est un choix pertinent face à ARIMA ou aux LSTM :

- **Données tabulaires et features riches** : notre pipeline produit des dizaines de features structurées (lags, moyennes mobiles, saison, ville, ...) que les modèles ensemblistes exploitent naturellement, là où ARIMA se limite à une seule série univariée et où un LSTM demande beaucoup de données et de réglages.
- **Rapidité d'entraînement** : XGBoost s'entraîne en quelques secondes sur des dizaines de milliers de lignes, ce qui facilite l'itération en TP.
- **Robustesse** : gestion native des valeurs manquantes, des variables catégorielles encodées et des relations non linéaires, avec régularisation intégrée contre le surapprentissage.
- **Interprétabilité** : l'importance des features (gain / SHAP) permet d'expliquer quelles variables (lags, saison, ville) pilotent la prévision, ce qui est précieux pour un TP pédagogique.

ARIMA resterait pertinent pour une série unique sans features exogènes ; un LSTM pourrait capter de longues dépendances mais exigerait un volume de données et un coût d'entraînement disproportionnés pour l'objectif visé.

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Échantillon limité (max 2000 lignes) pour un entraînement rapide.
sample = df.sort_values(["city", "timestamp"]).head(2000).copy()
sample["timestamp"] = pd.to_datetime(sample["timestamp"])

# Feature engineering rapide : lag1, moyenne mobile 7j, mois, jour de la semaine.
parts = []
for city, grp in sample.groupby("city"):
    grp = grp.sort_values("timestamp").copy()
    grp["lag1"] = grp["temperature"].shift(1)
    grp["ma7"] = grp["temperature"].rolling(7).mean()
    grp["month"] = grp["timestamp"].dt.month
    grp["dayofweek"] = grp["timestamp"].dt.dayofweek
    grp["target"] = grp["temperature"].shift(-1)
    parts.append(grp)
sample = pd.concat(parts, ignore_index=True)
sample = sample.dropna(subset=["target", "ma7", "lag1"])

feats = [c for c in ["lag1", "ma7", "month", "dayofweek", "precipitation", "wind_speed"] if c in sample.columns]
X = sample[feats]
y = sample["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

model = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
print("RMSE (échantillon) : {:.3f} °C".format(rmse))

## Conclusion

Ce notebook a permis de charger un échantillon Silver via WebHDFS, d'explorer les distributions (température, précipitations, vent) et d'entraîner un premier XGBoost avec un RMSE de référence. Dans le pipeline complet, ces étapes sont industrialisées dans `ml/feature_engineering.py` (features riches), `ml/train_model.py` (XGBoost + validation temporelle) et `ml/inference.py` (prévisions J+1 écrites en Gold).